# Solace Browser Deployment Notebook 06 — Ubuntu Snap Store

**Version:** 1.0.0 | **Auth:** 65537 | **Rung Target:** 65537

NORTHSTAR: build a snap package, smoke-test locally, upload to the Snap Store, promote through channels (edge -> candidate -> stable), and seal evidence with SHA-256 hash chain.

## Prerequisites

- `snapcraft` installed (`sudo snap install snapcraft --classic`)
- `snapcraft login` completed (Snap Store credentials)
- `snapcraft.yaml` in project root or `snap/` directory
- Snap name `solace-browser` registered (`snapcraft register solace-browser`)
- `VERSION` file in project root
- LXD or Multipass available for clean builds

In [ ]:
"""PREFLIGHT — verify snapcraft.yaml exists and snap name is registered."""
import os
import subprocess
import json
import hashlib
import re
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path('/home/phuc/projects/solace-browser')
VERSION = (PROJECT_ROOT / 'VERSION').read_text(encoding='utf-8').strip()
TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
EVIDENCE_DIR = PROJECT_ROOT / 'scratch' / 'snap-store-deploy' / TIMESTAMP
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

SNAP_NAME = 'solace-browser'

# --- Locate snapcraft.yaml ---
SNAPCRAFT_YAML = None
for candidate in [
    PROJECT_ROOT / 'snap' / 'snapcraft.yaml',
    PROJECT_ROOT / 'snapcraft.yaml',
    PROJECT_ROOT / 'snap' / 'snapcraft.yml',
]:
    if candidate.is_file():
        SNAPCRAFT_YAML = candidate
        break
assert SNAPCRAFT_YAML is not None, \
    'snapcraft.yaml not found in snap/ or project root'
print(f'snapcraft.yaml: {SNAPCRAFT_YAML}')

# --- Parse snapcraft.yaml for name and version ---
import yaml  # PyYAML
with SNAPCRAFT_YAML.open('r', encoding='utf-8') as f:
    snap_config = yaml.safe_load(f)

snap_declared_name = snap_config.get('name', '')
snap_declared_version = snap_config.get('version', '')
snap_grade = snap_config.get('grade', 'devel')
snap_confinement = snap_config.get('confinement', 'strict')

assert snap_declared_name == SNAP_NAME, \
    f'snapcraft.yaml name mismatch: {snap_declared_name!r} != {SNAP_NAME!r}'
print(f'Snap name: {snap_declared_name}')
print(f'Snap version: {snap_declared_version}')
print(f'Grade: {snap_grade}, Confinement: {snap_confinement}')

# --- Verify snapcraft is installed ---
snapcraft_version_result = subprocess.run(
    ['snapcraft', '--version'],
    capture_output=True, text=True, timeout=15
)
assert snapcraft_version_result.returncode == 0, \
    f'snapcraft not installed: {snapcraft_version_result.stderr}'
SNAPCRAFT_VERSION = snapcraft_version_result.stdout.strip()
print(f'snapcraft version: {SNAPCRAFT_VERSION}')

# --- Verify snap name is registered ---
whoami_result = subprocess.run(
    ['snapcraft', 'whoami'],
    capture_output=True, text=True, timeout=15
)
assert whoami_result.returncode == 0, \
    f'snapcraft not logged in: {whoami_result.stderr}'
print(f'snapcraft auth: {whoami_result.stdout.strip()}')

# Check registration by listing registered names
names_result = subprocess.run(
    ['snapcraft', 'list-registered'],
    capture_output=True, text=True, timeout=30
)
assert SNAP_NAME in names_result.stdout, \
    f'Snap name {SNAP_NAME!r} not registered. Run: snapcraft register {SNAP_NAME}'
print(f'Snap name {SNAP_NAME!r} is registered')

preflight = {
    'status': 'ok',
    'timestamp': TIMESTAMP,
    'version': VERSION,
    'snap_name': SNAP_NAME,
    'snap_declared_version': snap_declared_version,
    'snap_grade': snap_grade,
    'snap_confinement': snap_confinement,
    'snapcraft_version': SNAPCRAFT_VERSION,
    'snapcraft_yaml': str(SNAPCRAFT_YAML),
    'evidence_dir': str(EVIDENCE_DIR),
}
(EVIDENCE_DIR / 'preflight.json').write_text(
    json.dumps(preflight, indent=2), encoding='utf-8'
)
print(json.dumps(preflight, indent=2))

In [ ]:
"""BUILD — run snapcraft to produce the .snap file."""

# --- Clean previous build artifacts ---
clean_result = subprocess.run(
    ['snapcraft', 'clean'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=120
)
if clean_result.returncode == 0:
    print('snapcraft clean: OK')
else:
    print(f'snapcraft clean warning (non-fatal): {clean_result.stderr[:200]}')

# --- Build snap ---
# Use --destructive-mode on CI or --use-lxd for clean builds
BUILD_MODE = os.environ.get('SNAPCRAFT_BUILD_MODE', '--use-lxd')
build_cmd = ['snapcraft', BUILD_MODE]
if BUILD_MODE == '--destructive-mode':
    build_cmd = ['snapcraft', '--destructive-mode']
elif BUILD_MODE == '--use-lxd':
    build_cmd = ['snapcraft', '--use-lxd']
else:
    build_cmd = ['snapcraft']  # default: multipass

print(f'Build command: {" ".join(build_cmd)}')

build_result = subprocess.run(
    build_cmd,
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=1800  # 30 min max
)

build_log_path = EVIDENCE_DIR / 'snapcraft-build.log'
build_log_path.write_text(
    f'STDOUT:\n{build_result.stdout}\n\nSTDERR:\n{build_result.stderr}',
    encoding='utf-8'
)
assert build_result.returncode == 0, \
    f'snapcraft build failed (log: {build_log_path}): {build_result.stderr[-500:]}'
print('snapcraft build: OK')

# --- Locate the built .snap file ---
snap_candidates = list(PROJECT_ROOT.glob(f'{SNAP_NAME}_*.snap'))
assert len(snap_candidates) > 0, \
    f'No .snap file found matching {SNAP_NAME}_*.snap in {PROJECT_ROOT}'
# Pick the most recently modified
SNAP_PATH = max(snap_candidates, key=lambda p: p.stat().st_mtime)
print(f'Built snap: {SNAP_PATH.name}')

# --- Compute snap hash ---
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

SNAP_HASH = sha256_file(SNAP_PATH)
SNAP_SIZE = SNAP_PATH.stat().st_size

# --- Inspect snap metadata ---
inspect_result = subprocess.run(
    ['snap', 'info', '--verbose', str(SNAP_PATH)],
    capture_output=True, text=True, timeout=30
)
# Also use unsquashfs to verify internal structure
list_result = subprocess.run(
    ['unsquashfs', '-l', str(SNAP_PATH)],
    capture_output=True, text=True, timeout=30
)
snap_file_count = len(list_result.stdout.strip().split('\n')) if list_result.returncode == 0 else 0

build_evidence = {
    'status': 'ok',
    'snap_path': str(SNAP_PATH),
    'snap_name': SNAP_PATH.name,
    'snap_sha256': SNAP_HASH,
    'snap_size_bytes': SNAP_SIZE,
    'snap_size_mb': round(SNAP_SIZE / (1024 * 1024), 2),
    'snap_file_count': snap_file_count,
    'version': VERSION,
    'build_mode': BUILD_MODE,
}
(EVIDENCE_DIR / 'build-evidence.json').write_text(
    json.dumps(build_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(build_evidence, indent=2))

In [ ]:
"""TEST — install snap locally and run smoke tests."""

# --- Install snap locally (dangerous mode for unsigned local snaps) ---
install_result = subprocess.run(
    ['sudo', 'snap', 'install', '--dangerous', str(SNAP_PATH)],
    capture_output=True, text=True, timeout=120
)
assert install_result.returncode == 0, \
    f'snap install failed: {install_result.stderr}'
print(f'Installed: {install_result.stdout.strip()}')

# --- Verify snap is installed ---
list_result = subprocess.run(
    ['snap', 'list', SNAP_NAME],
    capture_output=True, text=True, timeout=15
)
assert list_result.returncode == 0, \
    f'snap not found after install: {list_result.stderr}'
print(f'snap list:\n{list_result.stdout.strip()}')

# --- Smoke test: check binary exists and runs ---
smoke_tests = []

# Test 1: version flag
version_result = subprocess.run(
    [SNAP_NAME, '--version'],
    capture_output=True, text=True, timeout=15
)
smoke_tests.append({
    'test': 'version_flag',
    'passed': version_result.returncode == 0,
    'output': version_result.stdout.strip()[:200],
})
print(f'Smoke test --version: {"PASS" if version_result.returncode == 0 else "FAIL"}')

# Test 2: help flag
help_result = subprocess.run(
    [SNAP_NAME, '--help'],
    capture_output=True, text=True, timeout=15
)
smoke_tests.append({
    'test': 'help_flag',
    'passed': help_result.returncode == 0,
    'output': help_result.stdout.strip()[:200],
})
print(f'Smoke test --help: {"PASS" if help_result.returncode == 0 else "FAIL"}')

# Test 3: snap connections (interfaces)
connections_result = subprocess.run(
    ['snap', 'connections', SNAP_NAME],
    capture_output=True, text=True, timeout=15
)
smoke_tests.append({
    'test': 'snap_connections',
    'passed': connections_result.returncode == 0,
    'output': connections_result.stdout.strip()[:500],
})
print(f'Snap connections:\n{connections_result.stdout.strip()}')

# Test 4: snap binary is in PATH
which_result = subprocess.run(
    ['which', SNAP_NAME],
    capture_output=True, text=True, timeout=10
)
smoke_tests.append({
    'test': 'binary_in_path',
    'passed': which_result.returncode == 0,
    'output': which_result.stdout.strip(),
})
print(f'Binary path: {which_result.stdout.strip()}')

# --- Remove local snap ---
remove_result = subprocess.run(
    ['sudo', 'snap', 'remove', SNAP_NAME],
    capture_output=True, text=True, timeout=60
)
print(f'Snap removed: {remove_result.stdout.strip()}')

# --- Assert all smoke tests passed ---
all_passed = all(t['passed'] for t in smoke_tests)
assert all_passed, \
    f'Smoke tests failed: {[t["test"] for t in smoke_tests if not t["passed"]]}'

test_evidence = {
    'status': 'ok',
    'smoke_tests': smoke_tests,
    'all_passed': all_passed,
    'test_count': len(smoke_tests),
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
(EVIDENCE_DIR / 'test-evidence.json').write_text(
    json.dumps(test_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(test_evidence, indent=2))

In [ ]:
"""UPLOAD — upload snap to the Snap Store and release to edge channel."""

# --- Upload to Snap Store ---
upload_result = subprocess.run(
    ['snapcraft', 'upload', str(SNAP_PATH), '--release=edge'],
    capture_output=True, text=True, timeout=600
)

upload_log_path = EVIDENCE_DIR / 'snapcraft-upload.log'
upload_log_path.write_text(
    f'STDOUT:\n{upload_result.stdout}\n\nSTDERR:\n{upload_result.stderr}',
    encoding='utf-8'
)
assert upload_result.returncode == 0, \
    f'snapcraft upload failed (log: {upload_log_path}): {upload_result.stderr[-500:]}'
print(f'Upload output:\n{upload_result.stdout.strip()}')

# --- Parse revision number from upload output ---
# Output typically: "Revision 42 created for 'solace-browser'"
revision_match = re.search(r'[Rr]evision\s+(\d+)', upload_result.stdout)
assert revision_match, \
    f'Could not parse revision from upload output: {upload_result.stdout[:300]}'
REVISION = int(revision_match.group(1))
print(f'Uploaded revision: {REVISION}')

# --- Verify edge channel has the revision ---
status_result = subprocess.run(
    ['snapcraft', 'status', SNAP_NAME],
    capture_output=True, text=True, timeout=30
)
assert status_result.returncode == 0, \
    f'snapcraft status failed: {status_result.stderr}'
print(f'Channel status:\n{status_result.stdout.strip()}')

# Verify revision is on edge
assert str(REVISION) in status_result.stdout, \
    f'Revision {REVISION} not found in channel status'

upload_evidence = {
    'status': 'ok',
    'revision': REVISION,
    'channel': 'edge',
    'snap_name': SNAP_NAME,
    'snap_file': SNAP_PATH.name,
    'channel_status': status_result.stdout.strip(),
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
(EVIDENCE_DIR / 'upload-evidence.json').write_text(
    json.dumps(upload_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(upload_evidence, indent=2))

In [ ]:
"""PROMOTE — release snap from edge -> candidate -> stable.

Each promotion step is verified before proceeding.
Set SOLACE_SNAP_PROMOTE_TARGET to control how far to promote.
"""

PROMOTION_LADDER = [
    ('edge', 'candidate'),
    ('candidate', 'stable'),
]

# --- Set promotion target ---
TARGET_CHANNEL = os.environ.get('SOLACE_SNAP_PROMOTE_TARGET', 'candidate')
print(f'Promotion target: {TARGET_CHANNEL}')

promotion_log = []

for from_channel, to_channel in PROMOTION_LADDER:
    print(f'\n--- Promoting {from_channel} -> {to_channel} ---')

    release_result = subprocess.run(
        ['snapcraft', 'release', SNAP_NAME, str(REVISION), to_channel],
        capture_output=True, text=True, timeout=60
    )
    assert release_result.returncode == 0, \
        f'snapcraft release to {to_channel} failed: {release_result.stderr}'
    print(f'Released revision {REVISION} to {to_channel}')
    print(f'Output: {release_result.stdout.strip()}')

    # Verify channel status after promotion
    verify_result = subprocess.run(
        ['snapcraft', 'status', SNAP_NAME],
        capture_output=True, text=True, timeout=30
    )
    assert verify_result.returncode == 0, \
        f'snapcraft status failed after promotion: {verify_result.stderr}'

    step = {
        'from': from_channel,
        'to': to_channel,
        'revision': REVISION,
        'channel_status': verify_result.stdout.strip(),
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
    promotion_log.append(step)
    print(f'Verified: {json.dumps(step, indent=2)}')

    if to_channel == TARGET_CHANNEL:
        print(f'\nReached target channel: {TARGET_CHANNEL}. Stopping.')
        break

(EVIDENCE_DIR / 'promotion-log.json').write_text(
    json.dumps(promotion_log, indent=2), encoding='utf-8'
)
print(f'\nPromotion log saved: {len(promotion_log)} steps')

In [ ]:
"""VERIFY — confirm snap info shows the latest version on the target channel."""

# --- Check snap info from the store ---
info_result = subprocess.run(
    ['snap', 'info', SNAP_NAME],
    capture_output=True, text=True, timeout=30
)
assert info_result.returncode == 0, \
    f'snap info failed: {info_result.stderr}'
print(f'snap info {SNAP_NAME}:\n{info_result.stdout.strip()}')

# --- Verify version appears in target channel ---
# Parse snap info output for channel lines
snap_info_output = info_result.stdout

# Check that target channel shows our version or revision
channel_pattern = re.compile(
    rf'^\s*{re.escape(TARGET_CHANNEL)}:\s+.*$', re.MULTILINE
)
channel_match = channel_pattern.search(snap_info_output)
assert channel_match, \
    f'Target channel {TARGET_CHANNEL!r} not found in snap info output'
channel_line = channel_match.group(0).strip()
print(f'Target channel line: {channel_line}')

# --- Verify via snapcraft status ---
final_status = subprocess.run(
    ['snapcraft', 'status', SNAP_NAME],
    capture_output=True, text=True, timeout=30
)
assert final_status.returncode == 0, \
    f'snapcraft status failed: {final_status.stderr}'
print(f'\nFinal channel status:\n{final_status.stdout.strip()}')

# --- Verify installability from store ---
# Install from the target channel to prove it works end-to-end
if TARGET_CHANNEL == 'stable':
    install_cmd = ['sudo', 'snap', 'install', SNAP_NAME]
else:
    install_cmd = ['sudo', 'snap', 'install', SNAP_NAME,
                   f'--channel={TARGET_CHANNEL}']

store_install_result = subprocess.run(
    install_cmd,
    capture_output=True, text=True, timeout=120
)
if store_install_result.returncode == 0:
    print(f'Store install from {TARGET_CHANNEL}: OK')
    # Quick version check
    v_check = subprocess.run(
        [SNAP_NAME, '--version'],
        capture_output=True, text=True, timeout=15
    )
    print(f'Installed version output: {v_check.stdout.strip()}')
    # Clean up
    subprocess.run(
        ['sudo', 'snap', 'remove', SNAP_NAME],
        capture_output=True, text=True, timeout=60
    )
    store_install_verified = True
else:
    print(f'Store install check failed (non-fatal): {store_install_result.stderr[:200]}')
    store_install_verified = False

verify_evidence = {
    'status': 'ok',
    'snap_name': SNAP_NAME,
    'target_channel': TARGET_CHANNEL,
    'channel_line': channel_line,
    'revision': REVISION,
    'store_install_verified': store_install_verified,
    'final_status': final_status.stdout.strip(),
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
(EVIDENCE_DIR / 'verify-evidence.json').write_text(
    json.dumps(verify_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(verify_evidence, indent=2))

In [ ]:
"""EVIDENCE SEAL — SHA-256 report with version, revision, snap hash.

Produces a tamper-evident evidence chain linking every artifact
from this deployment session.
"""

# --- Collect all evidence file hashes ---
evidence_files = sorted(EVIDENCE_DIR.glob('*.json'))
evidence_hashes = {}
for ef in evidence_files:
    evidence_hashes[ef.name] = sha256_file(ef)

# --- Build hash chain ---
chain_input = '\n'.join(
    f'{h}  {name}' for name, h in sorted(evidence_hashes.items())
)
chain_hash = hashlib.sha256(chain_input.encode('utf-8')).hexdigest()

# --- Git state ---
git_sha = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=10
).stdout.strip()

git_dirty = subprocess.run(
    ['git', 'status', '--porcelain'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=10
).stdout.strip()

seal = {
    'seal': 'SOLACE-SNAP-DEPLOY',
    'auth': 65537,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'version': VERSION,
    'snap_name': SNAP_NAME,
    'snap_file': SNAP_PATH.name,
    'snap_sha256': SNAP_HASH,
    'snap_size_bytes': SNAP_SIZE,
    'revision': REVISION,
    'target_channel': TARGET_CHANNEL,
    'snap_grade': snap_grade,
    'snap_confinement': snap_confinement,
    'snapcraft_version': SNAPCRAFT_VERSION,
    'git_sha': git_sha,
    'git_dirty': bool(git_dirty),
    'evidence_chain': {
        'files': evidence_hashes,
        'chain_hash': chain_hash,
    },
}

seal_path = EVIDENCE_DIR / 'evidence-seal.json'
seal_path.write_text(json.dumps(seal, indent=2), encoding='utf-8')

# --- Write human-readable report ---
report_lines = [
    f'# Snap Store Deploy Evidence — v{VERSION}',
    f'',
    f'**Timestamp:** {seal["timestamp"]}',
    f'**Auth:** {seal["auth"]}',
    f'**Snap Name:** {SNAP_NAME}',
    f'**Version:** v{VERSION}',
    f'**Revision:** {REVISION}',
    f'**Target Channel:** {TARGET_CHANNEL}',
    f'**Grade:** {snap_grade} | **Confinement:** {snap_confinement}',
    f'**Snapcraft:** {SNAPCRAFT_VERSION}',
    f'**Git SHA:** {git_sha}',
    f'**Git Dirty:** {bool(git_dirty)}',
    f'',
    f'## Snap Package',
    f'- **File:** {SNAP_PATH.name}',
    f'- **SHA-256:** `{SNAP_HASH}`',
    f'- **Size:** {SNAP_SIZE:,} bytes ({round(SNAP_SIZE / (1024*1024), 2)} MB)',
    f'',
    f'## Evidence Chain',
    f'- **Chain Hash:** `{chain_hash}`',
    f'',
    f'### Evidence Files',
]
for name, h in sorted(evidence_hashes.items()):
    report_lines.append(f'- `{h}  {name}`')

report_path = EVIDENCE_DIR / 'deploy-report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print(f'Evidence sealed: {seal_path}')
print(f'Report written: {report_path}')
print(f'Chain hash: {chain_hash}')
print(json.dumps(seal, indent=2))